# Catalan Accent Oracle: Colab / Kaggle Notebook

End-to-end pipeline for the Catalan Accent Oracle research prototype: CV26 metadata handling, selective audio extraction, HuBERT embeddings, baseline training, calibrated SVM artifact export, and optional held-out evaluation.

This notebook mirrors the repository scripts under `scripts/` and defaults to a quota-friendly smoke run (25 speakers per dialect, 2 clips per speaker). Switch to `full_1440` to reproduce the 1,440-clip training setup in `reports/cv26_train_1440_experiment.md`.

**Expected inputs**
- A clone or uploaded copy of this repository.
- The Common Voice 26 Catalan archive (`common-voice-scripted-speech-26-0-catala-fe69b989.tar.gz`), uploaded or mounted manually.

No secrets or tokens are required. GPU runtimes speed up embedding extraction; CPU fallback works for smoke tests.


## 1. Install Dependencies

Colab and Kaggle usually include many scientific packages, but the exact versions vary. Run this cell once per fresh runtime. Restart the kernel only if the notebook asks you to after package installation.


In [ ]:
%pip install -q -r requirements.txt


## 2. Configure Environment And Paths

Set `RUN_MODE = "smoke"` for a quick end-to-end check. Change it to `"full_1440"` when you are ready to reproduce the 1,440-clip experiment described in `reports/cv26_train_1440_experiment.md`.

For Colab, you can optionally mount Google Drive by setting `MOUNT_GOOGLE_DRIVE = True`. Kaggle users usually place uploaded files under `/kaggle/input` and write outputs under `/kaggle/working`.


In [ ]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

import pandas as pd
import torch

RUN_MODE = "smoke"  # "smoke" or "full_1440"
MOUNT_GOOGLE_DRIVE = False

REPO_DIR = Path.cwd()
ARCHIVE_NAME = "common-voice-scripted-speech-26-0-catala-fe69b989.tar.gz"

# Edit these if your cloud upload/mount uses a different location.
ARCHIVE_CANDIDATES = [
    REPO_DIR / "data/raw" / ARCHIVE_NAME,
    Path("/content/drive/MyDrive/proj-accents") / ARCHIVE_NAME,
    Path("/content") / ARCHIVE_NAME,
    Path("/kaggle/input") / ARCHIVE_NAME,
]

OUTPUT_ROOT = Path("/kaggle/working/proj-accents") if Path("/kaggle/working").exists() else REPO_DIR
METADATA_DIR = REPO_DIR / "data/metadata/cv26-ca"

if MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as exc:
        print(f"Google Drive mount skipped: {type(exc).__name__}: {exc}")

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Repo:", REPO_DIR)
print("Output root:", OUTPUT_ROOT)
print("CUDA available:", torch.cuda.is_available())


## 3. Locate Archive And Verify Metadata

The manifest script only needs TSV metadata. If `data/metadata/cv26-ca/train.tsv` is missing but the tarball is available, the next cell extracts only TSV/README files from the archive, not audio.


In [ ]:
def find_archive(candidates: list[Path]) -> Path | None:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None

archive_path = find_archive(ARCHIVE_CANDIDATES)
print("Archive:", archive_path if archive_path else "not found")

required_scripts = [
    "scripts/build_cv26_balanced_manifest.py",
    "scripts/prepare_cv26_audio_from_archive.py",
    "scripts/extract_hubert_embeddings.py",
    "scripts/train_embedding_baselines.py",
    "scripts/train_embedding_model_artifact.py",
]
missing_scripts = [path for path in required_scripts if not (REPO_DIR / path).exists()]
if missing_scripts:
    raise FileNotFoundError(f"Missing expected repository scripts: {missing_scripts}")

if not archive_path and not (METADATA_DIR / "train.tsv").exists():
    raise FileNotFoundError(
        "Upload or mount the CV26 Catalan tar.gz archive, or provide data/metadata/cv26-ca/train.tsv."
    )


In [ ]:
def extract_cv26_metadata_only(archive: Path, metadata_dir: Path) -> None:
    metadata_dir.mkdir(parents=True, exist_ok=True)
    wanted_suffixes = (".tsv", ".txt", ".md")
    extracted = []
    with tarfile.open(archive, "r:gz") as tar:
        for member in tar:
            name = member.name
            if "/ca/" not in name or not name.lower().endswith(wanted_suffixes):
                continue
            if "/clips/" in name:
                continue
            source = tar.extractfile(member) if member.isfile() else None
            if source is None:
                continue
            target = metadata_dir / Path(name).name
            with target.open("wb") as fh:
                shutil.copyfileobj(source, fh)
            extracted.append(target.name)
    print(f"Extracted metadata files: {sorted(set(extracted))}")

if not (METADATA_DIR / "train.tsv").exists():
    extract_cv26_metadata_only(archive_path, METADATA_DIR)

for split in ["train", "dev", "test"]:
    path = METADATA_DIR / f"{split}.tsv"
    print(split, "exists=" + str(path.exists()), "size=" + (str(path.stat().st_size) if path.exists() else "n/a"))

if not (METADATA_DIR / "train.tsv").exists():
    raise FileNotFoundError("train.tsv is still missing after metadata extraction.")


## 4. Choose Experiment Size

Smoke mode is intended to finish quickly and catch path or package problems. The full mode matches the local 1,440-clip run: after reserved speaker exclusion, it should select 96 speakers per dialect and 3 clips per speaker.


In [ ]:
if RUN_MODE == "smoke":
    MAX_SPEAKERS_PER_LABEL = 25
    MAX_CLIPS_PER_SPEAKER = 2
    RUN_TAG = "cv26-cloud-smoke"
    EMBEDDING_MAX_ROWS = None
elif RUN_MODE == "full_1440":
    MAX_SPEAKERS_PER_LABEL = 150
    MAX_CLIPS_PER_SPEAKER = 3
    RUN_TAG = "cv26-train-1440-cloud"
    EMBEDDING_MAX_ROWS = None
else:
    raise ValueError("RUN_MODE must be 'smoke' or 'full_1440'")

MANIFEST_PATH = OUTPUT_ROOT / "manifests" / f"{RUN_TAG}.csv"
AUDIO_DIR = OUTPUT_ROOT / "data/audio" / RUN_TAG
EMBEDDINGS_DIR = OUTPUT_ROOT / "embeddings" / RUN_TAG
BASELINE_DIR = OUTPUT_ROOT / "reports/baselines" / RUN_TAG
MODEL_DIR = OUTPUT_ROOT / "models" / f"{RUN_TAG}-hubert-svm-calibrated"

print(json.dumps({
    "run_mode": RUN_MODE,
    "manifest": str(MANIFEST_PATH),
    "audio_dir": str(AUDIO_DIR),
    "embeddings_dir": str(EMBEDDINGS_DIR),
    "baseline_dir": str(BASELINE_DIR),
    "model_dir": str(MODEL_DIR),
}, indent=2))


## 5. Build A Balanced Manifest

This reads only `train.tsv`, maps CV26 labels into the five macro dialect zones, excludes ambiguous speakers, and avoids speakers already reserved in `manifests/benchmark.csv` when that file is present.


In [ ]:
def run_cmd(args: list[str]) -> None:
    print("$", " ".join(args))
    subprocess.run(args, cwd=REPO_DIR, check=True)

run_cmd([
    sys.executable,
    "scripts/build_cv26_balanced_manifest.py",
    "--metadata-dir", str(METADATA_DIR),
    "--source-split", "train",
    "--out-manifest", str(MANIFEST_PATH),
    "--max-speakers-per-label", str(MAX_SPEAKERS_PER_LABEL),
    "--max-clips-per-speaker", str(MAX_CLIPS_PER_SPEAKER),
    "--seed", "13",
])

manifest = pd.read_csv(MANIFEST_PATH)
print("Rows:", len(manifest))
print("Speakers:", manifest["client_id"].nunique())
display(manifest.groupby("label").agg(rows=("path", "count"), speakers=("client_id", "nunique")))
manifest.head()


## 6. Extract Only Selected Audio

This scans the tarball and writes only the selected MP3 files from the manifest. The full archive is not extracted.


In [ ]:
if archive_path is None:
    raise FileNotFoundError("Audio extraction needs the CV26 tar.gz archive.")

run_cmd([
    sys.executable,
    "scripts/prepare_cv26_audio_from_archive.py",
    "--archive", str(archive_path),
    "--manifest", str(MANIFEST_PATH),
    "--out-dir", str(AUDIO_DIR),
])

prepared_manifest = AUDIO_DIR / "prepared_manifest.csv"
prepared = pd.read_csv(prepared_manifest)
print("Prepared rows:", int(prepared["audio_prepared"].sum()), "/", len(prepared))
prepared.head()


## 7. Extract HuBERT Embeddings

This is the most expensive step. It uses GPU when available and falls back to CPU otherwise. On a small smoke run it should be manageable; the full 1,440-clip run can take significantly longer depending on the runtime.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_cmd = [
    sys.executable,
    "scripts/extract_hubert_embeddings.py",
    "--prepared-manifest", str(prepared_manifest),
    "--out-dir", str(EMBEDDINGS_DIR),
    "--model-name", "BSC-LT/hubert-base-ca-2k",
    "--device", device,
    "--force-exit",
]
if EMBEDDING_MAX_ROWS is not None:
    embedding_cmd.extend(["--max-rows", str(EMBEDDING_MAX_ROWS)])

run_cmd(embedding_cmd)

embedding_index = EMBEDDINGS_DIR / "embedding_index.csv"
embeddings = pd.read_csv(embedding_index)
print("Embedded rows:", len(embeddings))
print("Embedding dim:", embeddings["embedding_dim"].dropna().iloc[0] if len(embeddings) else "n/a")
embeddings.head()


## 8. Train And Evaluate Baselines

The baseline script uses speaker-grouped cross-validation. For very small smoke runs it automatically reduces the fold count when needed.


In [ ]:
run_cmd([
    sys.executable,
    "scripts/train_embedding_baselines.py",
    "--embedding-index", str(embedding_index),
    "--out-dir", str(BASELINE_DIR),
])

results_path = BASELINE_DIR / "results.json"
results = json.loads(results_path.read_text(encoding="utf-8"))
pd.DataFrame(results["results"])[["model", "accuracy", "macro_f1", "top2_accuracy"]]


## 9. Save The Calibrated SVM Model Artifact

This trains the current best artifact shape on all embeddings from this run: StandardScaler plus calibrated linear SVM. The saved artifact can be copied back into the repository or used by the local inference API after matching paths and metadata.


In [ ]:
run_cmd([
    sys.executable,
    "scripts/train_embedding_model_artifact.py",
    "--train-index", str(embedding_index),
    "--out-dir", str(MODEL_DIR),
    "--encoder-model-name", "BSC-LT/hubert-base-ca-2k",
])

print("Model files:")
for path in sorted(MODEL_DIR.glob("*")):
    print("-", path, path.stat().st_size, "bytes")
json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))


## 10. Package Key Artifacts

Run this when you want a compact download containing the manifest, summaries, baseline reports, model artifact, and embedding index. Embedding vector files and audio clips can be large, so they are excluded by default.


In [ ]:
ARTIFACT_ZIP = OUTPUT_ROOT / f"{RUN_TAG}-key-artifacts.zip"
paths_to_zip = [
    MANIFEST_PATH,
    MANIFEST_PATH.with_suffix(".summary.md"),
    AUDIO_DIR / "summary.json",
    EMBEDDINGS_DIR / "summary.json",
    EMBEDDINGS_DIR / "embedding_index.csv",
    BASELINE_DIR / "results.json",
    BASELINE_DIR / "results.md",
    MODEL_DIR / "model.joblib",
    MODEL_DIR / "metadata.json",
]

import zipfile
with zipfile.ZipFile(ARTIFACT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in paths_to_zip:
        if path.exists():
            zf.write(path, arcname=str(path.relative_to(OUTPUT_ROOT) if path.is_relative_to(OUTPUT_ROOT) else path.name))

print("Wrote", ARTIFACT_ZIP, ARTIFACT_ZIP.stat().st_size, "bytes")

try:
    from google.colab import files
    # Uncomment to download automatically in Colab.
    # files.download(str(ARTIFACT_ZIP))
except Exception:
    pass


## Notes For Scaling Up

- Keep `RUN_MODE = "smoke"` until every step completes once in your runtime.
- Switch to `RUN_MODE = "full_1440"` to reproduce the strongest local training subset.
- Prefer GPU for embedding extraction. CPU fallback works, but it is slower.
- Do not combine CV26 official train/dev/test splits as independent data unless you are intentionally designing a new split policy.
- Keep held-out or manually recorded evaluation separate from the training manifest to avoid speaker leakage.
